# Topic 1: Spark Memory Model

> **Phase 4 → Week 6 → Topic 1**
>
> Prerequisites: Phase 3 complete (Weeks 3-5)

---

## The Office Desk Analogy

Think of an executor as a worker's desk in an office:

```
┌──────────────────────────── EXECUTOR MEMORY (e.g. 4GB) ───────────────────────────┐
│                                                                                    │
│  ┌─────────────────────────────────────────────────────┐  ┌────────────────────┐  │
│  │            JVM Heap  (spark.executor.memory)        │  │   Off-Heap         │  │
│  │                                                     │  │  (Tungsten)        │  │
│  │  ┌──────────────────────────────────────────────┐   │  │                    │  │
│  │  │      Spark Memory  (spark.memory.fraction    │   │  │  Encoded rows,     │  │
│  │  │           default 0.6 of heap)               │   │  │  sort buffers,     │  │
│  │  │                                              │   │  │  hash tables       │  │
│  │  │  ┌──────────────────┬───────────────────┐   │   │  │  (no GC overhead)  │  │
│  │  │  │ Execution Memory │  Storage Memory   │   │   │  │                    │  │
│  │  │  │  (shuffle, sort, │  (cache, persist) │   │   │  └────────────────────┘  │
│  │  │  │   aggregation)   │                   │   │   │                           │
│  │  │  │  ← borrow ──────→│                   │   │   │                           │
│  │  │  └──────────────────┴───────────────────┘   │   │                           │
│  │  └──────────────────────────────────────────────┘   │                           │
│  │                                                     │                           │
│  │  ┌──────────────────────────────────────────────┐   │                           │
│  │  │   User Memory  (1 - spark.memory.fraction)   │   │                           │
│  │  │   Python objects, UDF data, internal structs  │   │                           │
│  │  └──────────────────────────────────────────────┘   │                           │
│  │                                                     │                           │
│  │  ┌──────────────────────────────────────────────┐   │                           │
│  │  │   Reserved Memory  (300MB fixed)             │   │                           │
│  │  │   Spark internal objects                     │   │                           │
│  │  └──────────────────────────────────────────────┘   │                           │
│  └─────────────────────────────────────────────────────┘                           │
└────────────────────────────────────────────────────────────────────────────────────┘
```

---

## The Unified Memory Manager (Spark 1.6+)

**Key innovation**: Execution and Storage memory share the same pool and can **borrow from each other**.

```
spark.memory.fraction = 0.6   (60% of heap for Spark memory pool)
spark.memory.storageFraction = 0.5  (50% of Spark pool is initially storage)

Spark pool = (heap - 300MB reserved) × 0.6
Initial storage region = Spark pool × 0.5
Initial execution region = Spark pool × 0.5

Borrowing rules:
• Execution can evict storage cache if it needs more memory
• Storage can expand into unused execution memory
• Execution memory is NEVER evicted (would fail the task)
• Storage eviction = LRU cache eviction (data recomputed if needed)
```

---

## Memory Regions Summary

| Region | Purpose | Configurable | What fills it |
|--------|---------|-------------|---------------|
| **Reserved** | Spark internals | No (fixed 300MB) | Spark system objects |
| **User Memory** | App-level data | Via `memory.fraction` | Python objects, UDF vars, internal structs |
| **Storage Memory** | Cache / persist | Via `storageFraction` | `df.cache()`, `df.persist()` |
| **Execution Memory** | Compute ops | Via `memory.fraction` | Shuffles, sorts, aggregations |
| **Off-Heap** | Tungsten | `spark.memory.offHeap.*` | Encoded rows, binary buffers |

---

## Interview Cheat Sheet

**Q: What are the memory regions in a Spark executor?**
> An executor's JVM heap is split into: Reserved Memory (300MB fixed, Spark internals), User Memory (your Python objects/UDF data, ~40% of heap by default), and the Spark Memory Pool (~60%). The Spark pool is dynamically shared between Execution Memory (shuffles, sorts, aggregates) and Storage Memory (cache/persist). Execution can evict cached data when it needs more space.

**Q: What is the Unified Memory Manager?**
> Before Spark 1.6, execution and storage memory were statically partitioned — wasteful if one was full and the other empty. The Unified Memory Manager (default since 1.6) lets them borrow from each other. Execution memory can evict cached RDDs if needed; storage can use idle execution memory. Both share the same pool bounded by `spark.memory.fraction`.

**Q: What is Tungsten and off-heap memory?**
> Tungsten is Spark's physical execution engine. It uses off-heap (native) memory outside the JVM heap for storing binary-encoded rows, sort buffers, and hash tables. This bypasses JVM garbage collection entirely — no GC pauses on this memory. It also uses CPU cache-aware data layouts and whole-stage code generation for maximum speed.

**Q: What is `spark.memory.fraction`?**
> It's the fraction of (heap - 300MB reserved) that Spark uses for its execution + storage pool. Default 0.6. The remaining 0.4 is User Memory for Python objects and UDF data. Increasing it gives Spark more room for caching and complex operations but leaves less for Python overhead.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week6 - Memory Model") \
    .master("local[4]") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext
print(f"Spark {spark.version} ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:01:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 ready


## Part 1: Inspecting Memory Configuration

In [2]:
# Read current memory configuration
memory_configs = [
    "spark.executor.memory",
    "spark.driver.memory",
    "spark.executor.memoryOverhead",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
    "spark.memory.offHeap.enabled",
    "spark.memory.offHeap.size",
]

print("Current Memory Configuration:")
print("-" * 55)
for key in memory_configs:
    try:
        val = spark.conf.get(key)
    except Exception:
        val = "(not set — default)"
    print(f"  {key:<40s} = {val}")

Current Memory Configuration:
-------------------------------------------------------


  spark.executor.memory                    = 2g
  spark.driver.memory                      = 2g


  spark.executor.memoryOverhead            = (not set — default)
  spark.memory.fraction                    = (not set — default)
  spark.memory.storageFraction             = (not set — default)
  spark.memory.offHeap.enabled             = (not set — default)
  spark.memory.offHeap.size                = (not set — default)


In [3]:
# Calculate actual memory regions for a given executor memory
def memory_breakdown(executor_memory_mb: int, memory_fraction: float = 0.6,
                     storage_fraction: float = 0.5):
    reserved_mb = 300
    usable_mb   = executor_memory_mb - reserved_mb
    spark_pool  = usable_mb * memory_fraction
    user_mb     = usable_mb * (1 - memory_fraction)
    storage_mb  = spark_pool * storage_fraction
    execution_mb = spark_pool * (1 - storage_fraction)

    print(f"\nMemory Breakdown for executor.memory = {executor_memory_mb}MB")
    print(f"  Reserved (fixed):          {reserved_mb:>6} MB")
    print(f"  User Memory (Python/UDF):  {user_mb:>6.0f} MB  ({(1-memory_fraction)*100:.0f}% of usable)")
    print(f"  Spark Pool total:          {spark_pool:>6.0f} MB  ({memory_fraction*100:.0f}% of usable)")
    print(f"    → Initial Storage:       {storage_mb:>6.0f} MB  ({storage_fraction*100:.0f}% of Spark pool)")
    print(f"    → Initial Execution:     {execution_mb:>6.0f} MB  ({(1-storage_fraction)*100:.0f}% of Spark pool)")
    print(f"  [Both can borrow from each other dynamically]")

memory_breakdown(2048)    # 2GB executor
memory_breakdown(4096)    # 4GB executor
memory_breakdown(8192)    # 8GB executor


Memory Breakdown for executor.memory = 2048MB
  Reserved (fixed):             300 MB
  User Memory (Python/UDF):     699 MB  (40% of usable)
  Spark Pool total:            1049 MB  (60% of usable)
    → Initial Storage:          524 MB  (50% of Spark pool)
    → Initial Execution:        524 MB  (50% of Spark pool)
  [Both can borrow from each other dynamically]

Memory Breakdown for executor.memory = 4096MB
  Reserved (fixed):             300 MB
  User Memory (Python/UDF):    1518 MB  (40% of usable)
  Spark Pool total:            2278 MB  (60% of usable)
    → Initial Storage:         1139 MB  (50% of Spark pool)
    → Initial Execution:       1139 MB  (50% of Spark pool)
  [Both can borrow from each other dynamically]

Memory Breakdown for executor.memory = 8192MB
  Reserved (fixed):             300 MB
  User Memory (Python/UDF):    3157 MB  (40% of usable)
  Spark Pool total:            4735 MB  (60% of usable)
    → Initial Storage:         2368 MB  (50% of Spark pool)
    → Init

## Part 2: Execution Memory — Shuffle, Sort, Aggregation

In [4]:
# Execution memory is consumed by:
# 1. Shuffle write buffers (sort-merge shuffle)
# 2. Hash tables for aggregations (HashAggregate)
# 3. Sort buffers for ORDER BY, sortMergeJoin
# 4. Join hash maps (BroadcastHashJoin broadcast side)

# When execution memory is FULL → spill to disk
# Spill is recoverable (just slower) — OOM happens when even spill fails

print("""
Execution Memory Usage by Operation:
─────────────────────────────────────────────────────────────
Operation            Memory Used                Spill behavior
──────────────────   ────────────────────────   ──────────────
groupBy + agg        Hash table per partition   Spills partial
                                                agg to disk
orderBy / sort       Sort buffer                Spills sorted
                                                runs to disk
shuffle write        Sort + write buffers       Spills sort
                                                run to disk
shuffle read         Deserialization + hashing  Spills to disk
join (SMJ)           Sort buffers (both sides)  Spills to disk
join (BHJ)           Broadcast table in RAM     OOM if too large!
                     (no spill for broadcast)
─────────────────────────────────────────────────────────────
Rule: BroadcastHashJoin has NO spill mechanism — broadcast table
must fit in executor memory or you get OOM.
""")


Execution Memory Usage by Operation:
─────────────────────────────────────────────────────────────
Operation            Memory Used                Spill behavior
──────────────────   ────────────────────────   ──────────────
groupBy + agg        Hash table per partition   Spills partial
                                                agg to disk
orderBy / sort       Sort buffer                Spills sorted
                                                runs to disk
shuffle write        Sort + write buffers       Spills sort
                                                run to disk
shuffle read         Deserialization + hashing  Spills to disk
join (SMJ)           Sort buffers (both sides)  Spills to disk
join (BHJ)           Broadcast table in RAM     OOM if too large!
                     (no spill for broadcast)
─────────────────────────────────────────────────────────────
Rule: BroadcastHashJoin has NO spill mechanism — broadcast table
must fit in executor memory or you get OOM.

In [5]:
# Demonstrate spill — create data larger than available execution memory
# (In practice, trigger this by setting very low memory limits)
spark.conf.set("spark.sql.shuffle.partitions", "2")

# Generate data that will stress aggregation buffers
df = spark.range(500_000) \
    .withColumn("key",   (F.col("id") % 100).cast("string")) \
    .withColumn("value", F.rand() * 1000) \
    .withColumn("data",  F.repeat(F.lit("x"), 100))   # pad rows to use more memory

result = df.groupBy("key").agg(
    F.sum("value").alias("total"),
    F.count("*").alias("cnt"),
    F.avg("value").alias("avg_val")
)

result.show(5)
print(f"Result: {result.count()} groups")
print("\nCheck Spark UI → Stages → look for 'Spill (Memory)' and 'Spill (Disk)' columns")
print("If spill > 0, execution memory was exhausted and data was spilled to disk.")

spark.conf.set("spark.sql.shuffle.partitions", "4")

+---+------------------+----+------------------+
|key|             total| cnt|           avg_val|
+---+------------------+----+------------------+
|  1|2505474.7511278903|5000| 501.0949502255781|
|  3| 2457979.663170955|5000|  491.595932634191|
|  4|2476335.5331906583|5000| 495.2671066381317|
|  8| 2506310.313392019|5000|501.26206267840377|
| 12| 2501120.487045693|5000|500.22409740913866|
+---+------------------+----+------------------+
only showing top 5 rows



Result: 100 groups

Check Spark UI → Stages → look for 'Spill (Memory)' and 'Spill (Disk)' columns
If spill > 0, execution memory was exhausted and data was spilled to disk.


## Part 3: Storage Memory — Cache Eviction

In [6]:
from pyspark.storagelevel import StorageLevel

df1 = spark.range(1_000_000).withColumn("v", F.rand()).cache()
df2 = spark.range(1_000_000).withColumn("v", F.rand() * 2).cache()
df3 = spark.range(500_000).withColumn("v", F.rand() * 3).cache()

c1 = df1.count()
c2 = df2.count()
c3 = df3.count()

print(f"Cached 3 DataFrames: {c1:,}, {c2:,}, {c3:,} rows")
print("Go to Spark UI \u2192 Storage tab to see memory usage per cached RDD")
print("(statusTracker().getExecutorIds() removed \u2014 not available in PySpark 3.5)")


Cached 3 DataFrames: 1,000,000, 1,000,000, 500,000 rows
Go to Spark UI → Storage tab to see memory usage per cached RDD
(statusTracker().getExecutorInfos() removed — not available in PySpark 3.5)


In [7]:
# Clean up caches
df1.unpersist()
df2.unpersist()
df3.unpersist()
print("All caches released")

# Storage fraction tuning
print("""
Tuning spark.memory.storageFraction:
─────────────────────────────────────────────────────────
Default: 0.5 (50% of Spark pool protected for storage)

Increase to 0.7-0.8 when:
  • Iterative ML: same data cached and accessed many times
  • Small shuffles, large cache reuse

Decrease to 0.2-0.3 when:
  • ETL pipelines: data used once, no caching needed
  • Large shuffles, complex joins dominate
  • OOM during aggregations/joins (more execution memory)
─────────────────────────────────────────────────────────
""")

All caches released

Tuning spark.memory.storageFraction:
─────────────────────────────────────────────────────────
Default: 0.5 (50% of Spark pool protected for storage)

Increase to 0.7-0.8 when:
  • Iterative ML: same data cached and accessed many times
  • Small shuffles, large cache reuse

Decrease to 0.2-0.3 when:
  • ETL pipelines: data used once, no caching needed
  • Large shuffles, complex joins dominate
  • OOM during aggregations/joins (more execution memory)
─────────────────────────────────────────────────────────



## Part 4: Off-Heap Memory (Tungsten)

In [8]:
# Off-heap is disabled by default. Enable it when:
# - Long GC pauses are causing task timeouts
# - Large execution memory operations (sort, shuffle) cause GC thrashing
# - You have enough physical RAM beyond what you give JVM heap

print("""
Off-Heap Memory Configuration:
─────────────────────────────────────────────────────────────────────
spark.memory.offHeap.enabled = true
spark.memory.offHeap.size    = 2g

Total executor physical memory = executor.memory + offHeap.size + overhead

Example 4-core executor node with 16GB RAM:
  spark.executor.memory        = 8g   (JVM heap)
  spark.memory.offHeap.size    = 4g   (Tungsten native memory)
  spark.executor.memoryOverhead = 2g  (Python process, native libs)
  ─────────────────────────────────
  Total physical memory used   = 14g  (leave 2g for OS)

When to use off-heap:
  ✓ GC pauses > 1-2 seconds visible in Spark UI
  ✓ Tasks failing with java.lang.OutOfMemoryError: GC overhead limit
  ✓ Large in-memory sorts and joins
─────────────────────────────────────────────────────────────────────
""")


Off-Heap Memory Configuration:
─────────────────────────────────────────────────────────────────────
spark.memory.offHeap.enabled = true
spark.memory.offHeap.size    = 2g

Total executor physical memory = executor.memory + offHeap.size + overhead

Example 4-core executor node with 16GB RAM:
  spark.executor.memory        = 8g   (JVM heap)
  spark.memory.offHeap.size    = 4g   (Tungsten native memory)
  spark.executor.memoryOverhead = 2g  (Python process, native libs)
  ─────────────────────────────────
  Total physical memory used   = 14g  (leave 2g for OS)

When to use off-heap:
  ✓ GC pauses > 1-2 seconds visible in Spark UI
  ✓ Tasks failing with java.lang.OutOfMemoryError: GC overhead limit
  ✓ Large in-memory sorts and joins
─────────────────────────────────────────────────────────────────────



## Part 5: Memory Overhead — The Hidden Memory

Every executor needs memory BEYOND the JVM heap:

```
Total Physical Memory per Executor =
  spark.executor.memory          (JVM heap)
+ spark.executor.memoryOverhead  (non-heap overhead)

Default overhead = max(384MB, 0.1 × executor.memory)

Overhead is used for:
  • Python worker processes (PySpark)
  • JVM native code (JNI)
  • OS-level buffers, thread stacks
  • Arrow for Pandas UDFs

Common mistake: requesting 8GB executor on a 8GB node
→ actual memory = 8GB + 800MB overhead = 8.8GB
→ container/YARN kills the executor (exceeds node memory)

Rule: always leave room for overhead when sizing executors!
```

In [9]:
# Overhead calculation
def total_executor_memory(executor_memory_gb: float,
                          overhead_factor: float = 0.1,
                          min_overhead_mb: int = 384):
    heap_mb     = int(executor_memory_gb * 1024)
    overhead_mb = max(min_overhead_mb, int(heap_mb * overhead_factor))
    total_mb    = heap_mb + overhead_mb
    print(f"executor.memory={executor_memory_gb}g → "
          f"heap={heap_mb}MB + overhead={overhead_mb}MB = {total_mb}MB total")
    return total_mb

print("Physical memory required per executor:")
total_executor_memory(1)
total_executor_memory(4)
total_executor_memory(8)
total_executor_memory(16)

print()
print("On a 16GB node:")
for mem in [4, 6, 8, 12, 14]:
    total = total_executor_memory(mem)
    fits  = "✓ fits" if total <= 16384 else "✗ too large — YARN/K8s will kill!"
    print(f"  → {fits}")

Physical memory required per executor:
executor.memory=1g → heap=1024MB + overhead=384MB = 1408MB total
executor.memory=4g → heap=4096MB + overhead=409MB = 4505MB total
executor.memory=8g → heap=8192MB + overhead=819MB = 9011MB total
executor.memory=16g → heap=16384MB + overhead=1638MB = 18022MB total

On a 16GB node:
executor.memory=4g → heap=4096MB + overhead=409MB = 4505MB total
  → ✓ fits
executor.memory=6g → heap=6144MB + overhead=614MB = 6758MB total
  → ✓ fits
executor.memory=8g → heap=8192MB + overhead=819MB = 9011MB total
  → ✓ fits
executor.memory=12g → heap=12288MB + overhead=1228MB = 13516MB total
  → ✓ fits
executor.memory=14g → heap=14336MB + overhead=1433MB = 15769MB total
  → ✓ fits


## Exercises

1. For a cluster with 32GB RAM nodes, calculate the optimal `spark.executor.memory` value if you want to run 2 executors per node, leaving room for OS and overhead.
2. What is the Spark pool size for a 6GB executor with `memory.fraction=0.7`? How much is storage vs execution initially?
3. When would you increase `spark.memory.storageFraction` to 0.8? What's the risk?
4. A job keeps failing with `OutOfMemoryError` during a large sort. You have off-heap disabled. What are 3 things you can try?
5. Explain why Broadcast Hash Join can cause OOM but Sort Merge Join won't (it just slows down via spill).

In [10]:
# Exercise 1: Sizing for 32GB node, 2 executors per node
node_ram_mb = 32 * 1024
os_reserve_mb = 2 * 1024    # always leave 2GB for OS
executors_per_node = 2
overhead_pct = 0.1

available_mb = node_ram_mb - os_reserve_mb
per_executor_total_mb = available_mb // executors_per_node

# heap = total / (1 + overhead_factor)
heap_mb = int(per_executor_total_mb / (1 + overhead_pct))
overhead_mb = per_executor_total_mb - heap_mb

print(f"32GB node, 2 executors/node:")
print(f"  Available (after OS reserve): {available_mb}MB")
print(f"  Per executor (total):         {per_executor_total_mb}MB")
print(f"  spark.executor.memory:        {heap_mb}MB = {heap_mb/1024:.1f}g")
print(f"  Overhead:                     {overhead_mb}MB")
print(f"  → Use: spark.executor.memory={heap_mb//1024}g")

32GB node, 2 executors/node:
  Available (after OS reserve): 30720MB
  Per executor (total):         15360MB
  spark.executor.memory:        13963MB = 13.6g
  Overhead:                     1397MB
  → Use: spark.executor.memory=13g


In [11]:
# Exercise 2: Spark pool breakdown for 6GB with fraction=0.7
memory_breakdown(6 * 1024, memory_fraction=0.7, storage_fraction=0.5)


Memory Breakdown for executor.memory = 6144MB
  Reserved (fixed):             300 MB
  User Memory (Python/UDF):    1753 MB  (30% of usable)
  Spark Pool total:            4091 MB  (70% of usable)
    → Initial Storage:         2045 MB  (50% of Spark pool)
    → Initial Execution:       2045 MB  (50% of Spark pool)
  [Both can borrow from each other dynamically]
